# Model Evaluation Notebook — Agent Fact Detection System

This notebook performs **individual evaluation** of each pre-trained model used in the pipeline:

| # | Model | Task | Metric |
|---|-------|------|--------|
| 1 | `openai/whisper-base` | Speech-to-Text (ASR) | WER (Word Error Rate) |
| 2 | `cross-encoder/nli-deberta-v3-base` | Natural Language Inference | Accuracy, F1, Confusion Matrix |

> **Note:** The Brave Search API is a data retrieval tool, not a trainable model — no evaluation needed there.

---


---
## 1. Whisper ASR Evaluation

We evaluate the `openai/whisper-base` model using two datasets:

- **Mozilla Common Voice 11 (Spanish)** — Real-world multilingual speech, relevant to our use case.
- **LibriSpeech ASR (English, test-clean)** — The standard benchmark for clean read speech.

**Metric: WER (Word Error Rate)**

$$WER = \frac{S + D + I}{N}$$

Where `S`=substitutions, `D`=deletions, `I`=insertions, `N`=total words in reference.  
Lower is better (0% = perfect).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import whisper
import jiwer
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

# Setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_SAMPLES = 50  # Number of samples to evaluate (increase for more rigour)

print(f"Using device: {DEVICE.upper()}")
print(f"Loading Whisper 'base' model...")

model = whisper.load_model("base", device=DEVICE)
print("Whisper model loaded.")


### 1.1 Evaluation on Mozilla Common Voice (Spanish)

In [ ]:
print("Loading Common Voice (es) dataset — this may take a moment...")

ds_cv = load_dataset(
    "mozilla-foundation/common_voice_11_0",
    "es",
    split=f"test[:{N_SAMPLES}]",
    trust_remote_code=True
)
print(f"Loaded {len(ds_cv)} samples from Common Voice (es).")
ds_cv[0]


In [ ]:
def transcribe_audio_array(model, audio_array, sampling_rate=16000):
    """Transcribe a numpy audio array with Whisper."""
    import whisper
    # Whisper expects float32 mono at 16kHz
    audio = audio_array.astype(np.float32)
    # Resample if needed
    if sampling_rate != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sampling_rate, target_sr=16000)
    # Pad or trim to 30s window
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    _, probs = model.detect_language(mel)
    options = whisper.DecodingOptions(fp16=(DEVICE == "cuda"))
    result = whisper.decode(model, mel, options)
    return result.text

transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces(),
])


In [ ]:
references_cv = []
hypotheses_cv = []

print("Running inference on Common Voice (es)...")

for sample in tqdm(ds_cv, desc="Common Voice ES"):
    ref = sample["sentence"]
    audio_array = np.array(sample["audio"]["array"])
    sr = sample["audio"]["sampling_rate"]

    try:
        hyp = transcribe_audio_array(model, audio_array, sampling_rate=sr)
        references_cv.append(ref)
        hypotheses_cv.append(hyp)
    except Exception as e:
        print(f"Skipped sample: {e}")

wer_cv = jiwer.wer(references_cv, hypotheses_cv, reference_transform=transformation, hypothesis_transform=transformation)
cer_cv = jiwer.cer(references_cv, hypotheses_cv, reference_transform=transformation, hypothesis_transform=transformation)

print(f"\nCommon Voice (ES) Results:")
print(f"  WER : {wer_cv * 100:.2f}%")
print(f"  CER : {cer_cv * 100:.2f}%")


### 1.2 Evaluation on LibriSpeech (English, test-clean)

In [ ]:
print("Loading LibriSpeech test-clean dataset...")

ds_ls = load_dataset(
    "librispeech_asr",
    "clean",
    split=f"test[:{N_SAMPLES}]",
    trust_remote_code=True
)
print(f"Loaded {len(ds_ls)} samples from LibriSpeech test-clean.")


In [ ]:
references_ls = []
hypotheses_ls = []

print("Running inference on LibriSpeech test-clean...")

for sample in tqdm(ds_ls, desc="LibriSpeech EN"):
    ref = sample["text"]
    audio_array = np.array(sample["audio"]["array"])
    sr = sample["audio"]["sampling_rate"]

    try:
        hyp = transcribe_audio_array(model, audio_array, sampling_rate=sr)
        references_ls.append(ref)
        hypotheses_ls.append(hyp)
    except Exception as e:
        print(f"Skipped sample: {e}")

wer_ls = jiwer.wer(references_ls, hypotheses_ls, reference_transform=transformation, hypothesis_transform=transformation)
cer_ls = jiwer.cer(references_ls, hypotheses_ls, reference_transform=transformation, hypothesis_transform=transformation)

print(f"\nLibriSpeech (EN test-clean) Results:")
print(f"  WER : {wer_ls * 100:.2f}%")
print(f"  CER : {cer_ls * 100:.2f}%")


### 1.3 Whisper Results Summary & Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Summary Table ────────────────────────────────────────────────────────────
results_whisper = pd.DataFrame({
    "Dataset": ["Common Voice (ES)", "LibriSpeech (EN clean)"],
    "Samples": [len(references_cv), len(references_ls)],
    "WER (%)": [round(wer_cv * 100, 2), round(wer_ls * 100, 2)],
    "CER (%)": [round(cer_cv * 100, 2), round(cer_ls * 100, 2)],
})

print("=" * 50)
print("  WHISPER BASE — EVALUATION SUMMARY")
print("=" * 50)
display(results_whisper)

# ── Bar Chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results_whisper))
width = 0.35

bars1 = ax.bar(x - width/2, results_whisper["WER (%)"], width, label="WER (%)", color="#6366f1", alpha=0.85)
bars2 = ax.bar(x + width/2, results_whisper["CER (%)"], width, label="CER (%)", color="#2dd4bf", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(results_whisper["Dataset"])
ax.set_ylabel("Error Rate (%)")
ax.set_title("Whisper Base — WER & CER by Dataset", fontweight="bold")
ax.legend()
ax.bar_label(bars1, padding=3, fmt="%.1f%%")
ax.bar_label(bars2, padding=3, fmt="%.1f%%")
ax.set_ylim(0, max(results_whisper["WER (%)"].max(), results_whisper["CER (%)"].max()) * 1.3)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("whisper_evaluation.png", dpi=150)
plt.show()
print("Chart saved as whisper_evaluation.png")


---
## 2. NLI Cross-Encoder Evaluation

We evaluate `cross-encoder/nli-deberta-v3-base` on:

- **MultiNLI** (`multi_nli`, `validation_matched`) — Gold standard English NLI benchmark.
- **XNLI** (`xnli`, language `es`) — Multilingual NLI, key for our Spanish news use case.

**Metrics:** Accuracy, F1-Score (macro), Confusion Matrix.

**Label Mapping:**

| MultiNLI label | Our Label |
|---|---|
| 0 — entailment | entailment |
| 1 — neutral | neutral |
| 2 — contradiction | contradiction |


In [ ]:
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

NLI_N_SAMPLES = 200  # Number of samples per dataset

print(f"NLI model device: {DEVICE.upper()}")
print("Loading NLI Cross-Encoder...")
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-base", device=DEVICE)
print("NLI model loaded.")


### 2.1 Evaluation on MultiNLI (English, validation matched)

In [ ]:
print("Loading MultiNLI validation_matched...")

ds_mnli = load_dataset("multi_nli", split=f"validation_matched[:{NLI_N_SAMPLES}]")
print(f"Loaded {len(ds_mnli)} samples from MultiNLI.")

# MultiNLI labels: 0=entailment, 1=neutral, 2=contradiction
LABEL_MAP_MNLI = {0: "entailment", 1: "neutral", 2: "contradiction"}
# cross-encoder/nli-deberta-v3-base output order: contradiction(0), entailment(1), neutral(2)
CE_IDX_TO_LABEL = {0: "contradiction", 1: "entailment", 2: "neutral"}

pairs_mnli = [(row["premise"], row["hypothesis"]) for row in ds_mnli]
true_labels_mnli = [LABEL_MAP_MNLI[row["label"]] for row in ds_mnli]

print("Running NLI inference on MultiNLI...")
scores_mnli = nli_model.predict(pairs_mnli, show_progress_bar=True)
pred_labels_mnli = [CE_IDX_TO_LABEL[s.argmax()] for s in scores_mnli]

acc_mnli = accuracy_score(true_labels_mnli, pred_labels_mnli)
f1_mnli  = f1_score(true_labels_mnli, pred_labels_mnli, average="macro")

print(f"\nMultiNLI (EN) Results:")
print(f"Accuracy : {acc_mnli * 100:.2f}%")
print(f"F1 Macro : {f1_mnli * 100:.2f}%")
print("\n" + classification_report(true_labels_mnli, pred_labels_mnli))


### 2.2 Evaluation on XNLI (Spanish)

In [ ]:
print("Loading XNLI (es)...")

ds_xnli = load_dataset("xnli", "es", split=f"test[:{NLI_N_SAMPLES}]")
print(f"Loaded {len(ds_xnli)} samples from XNLI (es).")

# XNLI labels: 0=entailment, 1=neutral, 2=contradiction
LABEL_MAP_XNLI = {0: "entailment", 1: "neutral", 2: "contradiction"}

pairs_xnli = [(row["premise"], row["hypothesis"]) for row in ds_xnli]
true_labels_xnli = [LABEL_MAP_XNLI[row["label"]] for row in ds_xnli]

print("Running NLI inference on XNLI (es)...")
scores_xnli = nli_model.predict(pairs_xnli, show_progress_bar=True)
pred_labels_xnli = [CE_IDX_TO_LABEL[s.argmax()] for s in scores_xnli]

acc_xnli = accuracy_score(true_labels_xnli, pred_labels_xnli)
f1_xnli  = f1_score(true_labels_xnli, pred_labels_xnli, average="macro")

print(f"\XNLI (ES) Results:")
print(f"  Accuracy : {acc_xnli * 100:.2f}%")
print(f"  F1 Macro : {f1_xnli * 100:.2f}%")
print("\n" + classification_report(true_labels_xnli, pred_labels_xnli))


### 2.3 NLI Results Summary & Confusion Matrices

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
results_nli = pd.DataFrame({
    "Dataset": ["MultiNLI (EN matched)", "XNLI (ES)"],
    "Samples": [NLI_N_SAMPLES, NLI_N_SAMPLES],
    "Accuracy (%)": [round(acc_mnli * 100, 2), round(acc_xnli * 100, 2)],
    "F1 Macro (%)": [round(f1_mnli * 100, 2), round(f1_xnli * 100, 2)],
})

print("=" * 60)
print("  NLI CROSS-ENCODER DeBERTa-v3-base — EVALUATION SUMMARY")
print("=" * 60)
display(results_nli)

# ── Confusion Matrices ────────────────────────────────────────────────────────
LABELS = ["entailment", "neutral", "contradiction"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, true_l, pred_l, title in [
    (axes[0], true_labels_mnli, pred_labels_mnli, "MultiNLI (EN)"),
    (axes[1], true_labels_xnli, pred_labels_xnli, "XNLI (ES)"),
]:
    cm = confusion_matrix(true_l, pred_l, labels=LABELS)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABELS, yticklabels=LABELS,
        ax=ax, linewidths=0.5
    )
    ax.set_title(f"NLI Confusion Matrix\n{title}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.tight_layout()
plt.savefig("nli_confusion_matrices.png", dpi=150)
plt.show()
print("Chart saved as nli_confusion_matrices.png")


---
## 3. Global Summary

In [ ]:
print("=" * 65)
print("  AGENT FACT DETECTION SYSTEM — FULL EVALUATION SUMMARY")
print("=" * 65)

print("\nWHISPER BASE (ASR):")
display(results_whisper)

print("\nNLI CROSS-ENCODER DeBERTa-v3-base:")
display(results_nli)

# ── Combined visual summary ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Whisper WER bars
ax = axes[0]
colors = ["#6366f1", "#2dd4bf"]
bars = ax.bar(results_whisper["Dataset"], results_whisper["WER (%)"], color=colors, alpha=0.85)
ax.bar_label(bars, padding=3, fmt="%.1f%%", fontweight="bold")
ax.set_title("Whisper Base — WER (%)", fontweight="bold")
ax.set_ylabel("WER (%) — lower is better")
ax.set_ylim(0, max(results_whisper["WER (%)"]) * 1.4)
ax.grid(axis="y", alpha=0.3)

# NLI Accuracy bars
ax = axes[1]
bars = ax.bar(results_nli["Dataset"], results_nli["Accuracy (%)"], color=colors, alpha=0.85)
ax.bar_label(bars, padding=3, fmt="%.1f%%", fontweight="bold")
ax.set_title("NLI DeBERTa-v3-base — Accuracy (%)", fontweight="bold")
ax.set_ylabel("Accuracy (%) — higher is better")
ax.set_ylim(0, 110)
ax.grid(axis="y", alpha=0.3)

fig.suptitle("Agent Fact Detection System — Model Evaluation", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.savefig("full_evaluation_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n All evaluations completed! Charts saved.")
